
# 🧩 Matchkey Generator

This notebook reads an **orders.csv** file, generates a standardized **matchkey** based on address fields, and saves two outputs:

1. **orders_with_matchkey.csv** → original dataset + new column `matchkey`  
2. **orders_with_hashed_matchkey_sha256.csv** → same dataset but with `matchkey` replaced by its SHA-256 hash

Make sure the `orders.csv` file is placed in the same folder as this notebook before running.


In [19]:

# Step 1: Imports and configuration
import pandas as pd
import re
import hashlib
from pathlib import Path

INPUT_FILE = Path("orders.csv")
OUTPUT_FILE = Path("orders_with_matchkey.csv")
HASHED_OUTPUT_FILE = Path("orders_with_hashed_matchkey_sha256.csv")

print("✅ Libraries imported successfully")


✅ Libraries imported successfully


In [20]:

# Step 2: Check for input file existence
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File '{INPUT_FILE.name}' not found in the current directory.")

print(f"✅ Found file: {INPUT_FILE.name}")


✅ Found file: orders.csv


In [21]:

# Step 3: Read CSV file
df = pd.read_csv(INPUT_FILE)
print(f"✅ File loaded successfully with {len(df)} rows and {len(df.columns)} columns")
df.head()


✅ File loaded successfully with 3 rows and 9 columns


,id,source_ref,source_created_at,total_price,line1,line2,state,postal_code,country_code
0,1001,SHOPIFY-123,2024-09-01T14:25:00Z,54.90,123 Main St,Apt 4,CA,94105-1234,US
1,1002,SHOPIFY-124,2024-09-02T11:10:00Z,89.50,456 Oak Avenue,NaN,NY,10001,US
2,1003,SHOPIFY-125,2024-09-03T08:45:00Z,39.99,789 Pine Blvd,Unit 12,TX,73301,US


In [22]:

# Step 4: Validate required columns
required_cols = {"line1", "line2", "postal_code"}
df.columns = [c.lower() for c in df.columns]
missing = required_cols - set(df.columns)

if missing:
    raise ValueError(f"Missing required columns: {', '.join(missing)}")

print("✅ Required columns found")


✅ Required columns found


In [23]:

# Step 5: Define helper functions

def sanitize_string(value: str) -> str:
    if pd.isna(value):
        return ""
    value = re.sub(r"[\x00-\x1F\x7F]", "", str(value))
    value = re.sub(r"[^-A-Za-zÁÉÍÓÚÀÈÌÒÙÂÊÎÔÛÃÕÄËÏÖÜÑÇáéíóúàèìòùâêîôûãõäëïöüñç0-9/ ']", "", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip().upper()


def normalize_zip5(zip_code: str) -> str:
    if pd.isna(zip_code):
        zip_code = ""
    digits = re.sub(r"[^0-9]", "", str(zip_code))
    if len(digits) < 5:
        return digits.zfill(5)
    return digits[:5]


def generate_matchkey(row):
    line1 = sanitize_string(row.get("line1", ""))
    line2 = sanitize_string(row.get("line2", ""))
    zip5 = normalize_zip5(row.get("postal_code", ""))
    return (line1 + line2 + zip5).replace(" ", "")

print("✅ Functions loaded successfully")


✅ Functions loaded successfully


In [24]:

# Step 6: Generate matchkeys
df["matchkey"] = df.apply(generate_matchkey, axis=1)
print("✅ Matchkeys generated successfully")
df.head()


✅ Matchkeys generated successfully


,id,source_ref,source_created_at,total_price,line1,line2,state,postal_code,country_code,matchkey
0,1001,SHOPIFY-123,2024-09-01T14:25:00Z,54.90,123 Main St,Apt 4,CA,94105-1234,US,123MAINSTAPT494105
1,1002,SHOPIFY-124,2024-09-02T11:10:00Z,89.50,456 Oak Avenue,NaN,NY,10001,US,456OAKAVENUE10001
2,1003,SHOPIFY-125,2024-09-03T08:45:00Z,39.99,789 Pine Blvd,Unit 12,TX,73301,US,789PINEBLVDUNIT1273301


In [25]:

# Step 7: Export main output (with matchkey)
df.to_csv(OUTPUT_FILE, index=False)
print(f"✅ File '{OUTPUT_FILE.name}' created successfully ({len(df)} rows)")


✅ File 'orders_with_matchkey.csv' created successfully (3 rows)


In [26]:

# Step 8: Export hashed output (matchkey replaced with SHA-256 hash)
hashed_df = df.copy()
hashed_df["matchkey"] = hashed_df["matchkey"].astype(str).apply(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest())

hashed_df.to_csv(HASHED_OUTPUT_FILE, index=False)
print(f"✅ File '{HASHED_OUTPUT_FILE.name}' created successfully ({len(hashed_df)} rows)")


✅ File 'orders_with_hashed_matchkey_sha256.csv' created successfully (3 rows)



### ✅ Done!
Two files have been generated:
- **orders_with_matchkey.csv** → contains the original matchkey  
- **orders_with_hashed_matchkey_sha256.csv** → same dataset but with the matchkey hashed using SHA-256
